# StudyMate: export your fine-tuned Qwen to Ollama

Run this **after** Part B of `StudyMate_FINAL.ipynb` printed `use_adapter: true` and downloaded `studymate_adapter.zip`.
If Part B said "keep the base model", skip this notebook and use `Modelfile.base` instead; that is a valid, honest result.

Steps: upload adapter zip -> merge it into the base Qwen -> convert to GGUF (the format Ollama reads) -> save to Google Drive.

Runtime: Colab, **T4 GPU** is fine (CPU also works, slower). Takes about 10-15 minutes.

## 1. Install

In [ ]:
!pip -q install transformers peft accelerate sentencepiece protobuf gguf huggingface_hub

## 2. Upload `studymate_adapter.zip` (the file Part B downloaded)

In [ ]:
import os, glob, zipfile, shutil
from google.colab import files
up = files.upload()
zp = next(n for n in up if n.endswith('.zip'))
shutil.rmtree('adapter', ignore_errors=True)
zipfile.ZipFile(zp).extractall('adapter')
cfgs = glob.glob('adapter/**/adapter_config.json', recursive=True)
best = [c for c in cfgs if os.sep + 'best' + os.sep in c] or cfgs
assert best, 'adapter_config.json not found in the zip'
ADAPTER_DIR = os.path.dirname(best[0])
print('using adapter:', ADAPTER_DIR)

## 3. Merge the adapter into the base model (fp16)

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

BASE = 'Qwen/Qwen2.5-3B-Instruct'
tok = AutoTokenizer.from_pretrained(BASE)
base = AutoModelForCausalLM.from_pretrained(BASE, torch_dtype=torch.float16, device_map='auto')
merged = PeftModel.from_pretrained(base, ADAPTER_DIR).merge_and_unload()
merged.save_pretrained('merged', safe_serialization=True)
tok.save_pretrained('merged')
print('merged model saved to ./merged')
del merged, base
import gc; gc.collect(); torch.cuda.empty_cache()

## 4. Convert to GGUF (Q8_0: near-lossless, about 3.3 GB)

In [ ]:
!git clone --depth 1 https://github.com/ggml-org/llama.cpp
!python llama.cpp/convert_hf_to_gguf.py merged --outfile studymate-qwen-q8_0.gguf --outtype q8_0
!ls -lh studymate-qwen-q8_0.gguf

### Optional: a smaller Q4_K_M file (about 1.9 GB, faster on CPU, tiny quality loss)
Needs a build of llama.cpp (about 5-8 minutes). Skip it if the Q8_0 file is fine for you.
If you use it, change the `FROM` line in `Modelfile.tuned` to `studymate-qwen-q4_k_m.gguf`.

In [ ]:
# OPTIONAL
!cd llama.cpp && cmake -B build -DGGML_CUDA=OFF > /dev/null && cmake --build build --config Release -j --target llama-quantize > /dev/null
!llama.cpp/build/bin/llama-quantize studymate-qwen-q8_0.gguf studymate-qwen-q4_k_m.gguf Q4_K_M
!ls -lh *.gguf

## 5. Save to Google Drive (browser downloads of 3 GB files often fail)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
os.makedirs('/content/drive/MyDrive/studymate', exist_ok=True)
for f in glob.glob('*.gguf'):
    shutil.copy(f, '/content/drive/MyDrive/studymate/')
    print('copied', f)
print('Now download the .gguf from Google Drive > studymate to your computer.')

## 6. On your computer

1. Put the `.gguf` file and `Modelfile.tuned` in one folder.
2. `ollama create studymate-qwen -f Modelfile.tuned`
3. `ollama run studymate-qwen "Summarise: Polymorphism lets one interface have many forms."` (quick test)
4. In the backend `.env` set `SUMMARIZER_PROVIDER=OLLAMA_QWEN` and restart the backend.